In [1]:
df = spark.sql("SELECT * FROM fraud_lakehouse.dbo.silver_transactions LIMIT 1000")
display(df)

StatementMeta(, 6e165cce-f7d0-4d7b-b802-1bee1a8260a4, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e402ab25-9866-474a-87d7-b3229ba6e782)

In [2]:
df_silver = spark.table("silver_transactions")
print(f"Silver records: {df_silver.count()}")

StatementMeta(, 6e165cce-f7d0-4d7b-b802-1bee1a8260a4, 4, Finished, Available, Finished, False)

Silver records: 568630


In [3]:
from pyspark.sql.functions import col, count, sum as spark_sum, avg, round as spark_round, when

df_fraud_by_amount = df_silver.groupBy("Amount_category").agg(
    count("*").alias("total_transactions"),
    spark_sum(when(col("Class") == 1, 1).otherwise(0)).alias("fraud_count"),
    spark_sum(when(col("Class") == 0, 1).otherwise(0)).alias("legit_count"),
    spark_round(avg("Amount"), 2).alias("avg_amount"),
    spark_round(
        spark_sum(when(col("Class") == 1, 1).otherwise(0)) * 100.0 / count("*"), 4
    ).alias("fraud_rate_pct")
)

df_fraud_by_amount.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fraud_by_amount")

df_fraud_by_amount.show()

StatementMeta(, 6e165cce-f7d0-4d7b-b802-1bee1a8260a4, 5, Finished, Available, Finished, False)

+---------------+------------------+-----------+-----------+----------+--------------+
|Amount_category|total_transactions|fraud_count|legit_count|avg_amount|fraud_rate_pct|
+---------------+------------------+-----------+-----------+----------+--------------+
|           High|             18982|       9400|       9582|    596.09|       49.5206|
|      Very_High|            546120|     273190|     272930|  12516.78|       50.0238|
|         Medium|              3528|       1725|       1803|    125.31|       48.8946|
+---------------+------------------+-----------+-----------+----------+--------------+



In [4]:
df_fraud_by_quartile = df_silver.groupBy("Amount_quartile").agg(
    count("*").alias("total_transactions"),
    spark_sum(when(col("Class") == 1, 1).otherwise(0)).alias("fraud_count"),
    spark_round(avg("Amount"), 2).alias("avg_amount"),
    spark_round(
        spark_sum(when(col("Class") == 1, 1).otherwise(0)) * 100.0 / count("*"), 4
    ).alias("fraud_rate_pct")
).orderBy("Amount_quartile")

df_fraud_by_quartile.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fraud_by_quartile")

df_fraud_by_quartile.show()

StatementMeta(, 6e165cce-f7d0-4d7b-b802-1bee1a8260a4, 6, Finished, Available, Finished, False)

+---------------+------------------+-----------+----------+--------------+
|Amount_quartile|total_transactions|fraud_count|avg_amount|fraud_rate_pct|
+---------------+------------------+-----------+----------+--------------+
|              1|            142158|      70834|    3058.4|       49.8277|
|              2|            142158|      70924|   9041.79|        49.891|
|              3|            142157|      71518|  15032.66|       50.3092|
|              4|            142157|      71039|  21035.06|       49.9722|
+---------------+------------------+-----------+----------+--------------+



In [5]:
from pyspark.sql.functions import current_timestamp

df_kpis = df_silver.agg(
    count("*").alias("total_transactions"),
    spark_sum(when(col("Class") == 1, 1).otherwise(0)).alias("total_frauds"),
    spark_sum(when(col("Class") == 0, 1).otherwise(0)).alias("total_legit"),
    spark_round(
        spark_sum(when(col("Class") == 1, 1).otherwise(0)) * 100.0 / count("*"), 4
    ).alias("overall_fraud_rate"),
    spark_round(
        spark_sum(when(col("Class") == 1, col("Amount")).otherwise(0)), 2
    ).alias("total_fraud_amount"),
    spark_round(avg("Amount"), 2).alias("avg_transaction_amount")
).withColumn("report_generated_at", current_timestamp())

df_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_kpis")

df_kpis.show(truncate=False)

StatementMeta(, 6e165cce-f7d0-4d7b-b802-1bee1a8260a4, 7, Finished, Available, Finished, False)

+------------------+------------+-----------+------------------+------------------+----------------------+--------------------------+
|total_transactions|total_frauds|total_legit|overall_fraud_rate|total_fraud_amount|avg_transaction_amount|report_generated_at       |
+------------------+------------+-----------+------------------+------------------+----------------------+--------------------------+
|568630            |284315      |284315     |50.0              |3.42815704535E9   |12041.96              |2026-03-24 22:52:20.644876|
+------------------+------------+-----------+------------------+------------------+----------------------+--------------------------+



In [6]:
feature_cols_for_summary = [f"V{i}" for i in range(1, 29)] + ["Amount", "Amount_log"]

df_feature_summary = df_silver.groupBy("Class").agg(
    *[spark_round(avg(c), 4).alias(f"avg_{c}") for c in feature_cols_for_summary]
)

df_feature_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_feature_summary")

print("Gold feature summary table created")

StatementMeta(, 6e165cce-f7d0-4d7b-b802-1bee1a8260a4, 8, Finished, Available, Finished, False)

Gold feature summary table created
